![DB Academy](./Includes/images/db-academy.png)


<div style="max-width: 1000px; margin: 0 auto; font-family: sans-serif;">

<div style="background: #02A36F; color: white; border-radius: 8px; padding: 28px 32px; text-align: center; position: relative;">
  <div style="font-size: 14pt; font-weight: 600; text-transform: uppercase; letter-spacing: 1px; opacity: 0.85; margin-bottom: 8px;">Lab</div>
  <div style="font-size: 24pt; font-weight: 700; line-height: 1.3;">Build a Customer Growth Dashboard</div>
  <div style="font-size: 14pt; margin-top: 12px; opacity: 0.9;">Apply what you learned in Lesson 08 to build a fresh dashboard from a constrained spec, one Counter, one dual-axis chart, one Filter, published to your account.</div>
</div>

</div>

## REQUIRED: SELECT A COMPUTE ENVIRONMENT

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 14px 18px; border-radius: 4px; margin: 16px 0;">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select SQL Warehouse Compute</strong>
  <div style="color:#333;">

This notebook requires a <strong>SQL Warehouse</strong> for execution. Before running any cells, attach this notebook to a SQL Warehouse. Select the compute tile in the top-right corner of the notebook, then choose your SQL Warehouse from the list.
  </div>
</div>

## PREREQUISITE - ENVIRONMENT SETUP

<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">
    Required Setup
  </strong>
  <div style="color:#333;">
  Before running this notebook, make sure you have already executed the <strong>Required Setup</strong>, in the <strong>00 - Course Overview</strong> notebook, to properly setup your catalog, schema, and files for this notebook.
  </div>
</div>

### Setup
Run the cell below to configure your environment for this lesson. The setup script will initialize your catalog and schema so your queries run against the correct context.

In [0]:
%run ./Includes/Classroom-Setup

**Where we left off:** In Lesson 08 you watched a full dashboard build for the Retail Organization, Counter, Combo Chart, Genie-generated bar, Scatter, Filter. Now your **Marketing lead** has a smaller, focused ask:

> *"For the weekly review I want a Customer Growth dashboard with three things only, how many customers we have versus a 1,000-customer target, a monthly trend of revenue and order volume, and a state filter so I can spot regional differences."*

In this practice you'll build that dashboard from scratch. The constrained spec:

1. **One Counter**: `Total_Customers` versus a goal of 1,000.
2. **One dual-axis chart**: monthly revenue (bar) and monthly order count (line).
3. **One Filter**: on `state`, applied across the whole dashboard.
4. **Publish** the result with account-wide view access.

The two SQL datasets below have `<FILL_IN>` placeholders. **Run them in this notebook first** to confirm they return the right shape, then paste the working SQL into your dashboard's **Data** tab. Hints are collapsed under each task.

### Task 1: Create the dashboard and add a base dataset

1. In the left sidebar, click **Dashboards**, then **Create dashboard**.
2. Rename the dashboard to **Customer Growth Dashboard**.
3. Click the **Data** tab.
4. Click **+ Add data** and add the **`customers`** table from your `retail` schema. (You'll add SQL datasets in the next two tasks; this whole-table dataset gives the filter widget something to bind to.)

### Task 2: Counter: `Total_Customers` vs. goal

Write a SQL dataset that returns a **single row with two columns**:

- `Total_Customers`: the total row count from `customers`
- `Customer_Goal`: the literal value `25000`

Fill in the cell below and run it to verify the output. Once it works, copy the SQL into a new SQL dataset on your dashboard and rename the dataset to **Customer Count**.

In [0]:
%sql
-- TODO: Return one row with Total_Customers and Customer_Goal
SELECT
  <FILL_IN> AS Total_Customers,
  <FILL_IN> AS Customer_Goal
FROM customers;

##### ANSWER

<details>
  <summary>EXPAND FOR SOLUTION</summary>

  The version you want is the one with operation <code>UPDATE</code> in the <code>DESCRIBE HISTORY</code> output. If you ran the lab cleanly from the top, that's <strong>version 2</strong>:

<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
<!-------------------ADD SOLUTION CODE BELOW------------------->
SELECT
  COUNT(*) AS Total_Customers,
  25000     AS Customer_Goal
FROM customers;
<!-------------------END SOLUTION CODE------------------->
</code></pre>


<script>
function copyBlock() {
  const el = document.getElementById("copy-block");
  if (!el) return;

  const text = el.innerText;

  // Preferred modern API
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackCopy(text);
      });
  } else {
    fallbackCopy(text);
  }
}

function fallbackCopy(text) {
  const textarea = document.createElement("textarea");
  textarea.value = text;
  textarea.style.position = "fixed";
  textarea.style.left = "-9999px";
  document.body.appendChild(textarea);
  textarea.select();
  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } catch (err) {
    console.error("Fallback copy failed:", err);
    alert("Could not copy to clipboard. Please copy manually.");
  } finally {
    document.body.removeChild(textarea);
  }
}
</script>
</details>


You should see exactly **one row** with two columns. The `Total_Customers` value will reflect however many rows are in your `customers` table; `Customer_Goal` should be `25000`.

Now wire it up on the dashboard:

1. Switch to the **Untitled page** canvas tab and click **Add a visualization**, then click on the canvas to drop the widget.
2. In the **Configuration Panel**, set:
   - **Title:** *checked*. Rename it to **Customer Growth**.
   - **Visualization:** *Counter*
   - **Value:** *Customer Count* > *Total_Customers*
   - **Comparison:** *Customer Count* > *Customer_Goal*
3. Open the **Style** section and add a **Conditional Style**: *If Value ≤ Target → Color Red*. *If Value > Target → Color Green*. The counter goes red when you're at or below goal and green when you're above it.

### Task 3: Dual-axis chart: monthly revenue and order count

Write a second SQL dataset that returns one row per month with three columns:

- `Month`: the order month formatted as `yyyy-MM` (use `date_format(order_date, 'yyyy-MM')`)
- `Monthly_Revenue`: the sum of `total_price` for that month
- `Order_Count`: the count of rows for that month

Sort by `Month` ascending. Select your catalog (`learn_get_started_da_YOUR_USERNAME`) and schema (`retail`). Run it to verify, then add it to the dashboard as a SQL dataset named **Monthly Trend**.

In [0]:
%sql
-- TODO: One row per month with Monthly_Revenue and Order_Count
SELECT
  date_format(order_date, 'yyyy-MM') AS Month,
  <FILL_IN>(total_price) AS Monthly_Revenue,
  <FILL_IN>              AS Order_Count
FROM sales
GROUP BY <FILL_IN>
ORDER BY <FILL_IN>;

##### ANSWER

<details>
  <summary>EXPAND FOR SOLUTION</summary>

  The version you want is the one with operation <code>UPDATE</code> in the <code>DESCRIBE HISTORY</code> output. If you ran the lab cleanly from the top, that's <strong>version 2</strong>:

<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
<!-------------------ADD SOLUTION CODE BELOW------------------->
SELECT
  date_format(order_date, 'yyyy-MM') AS Month,
  SUM(total_price)                   AS Monthly_Revenue,
  COUNT(*)                           AS Order_Count
FROM sales
GROUP BY Month
ORDER BY Month ASC;
<!-------------------END SOLUTION CODE------------------->
</code></pre>


<script>
function copyBlock() {
  const el = document.getElementById("copy-block");
  if (!el) return;

  const text = el.innerText;

  // Preferred modern API
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackCopy(text);
      });
  } else {
    fallbackCopy(text);
  }
}

function fallbackCopy(text) {
  const textarea = document.createElement("textarea");
  textarea.value = text;
  textarea.style.position = "fixed";
  textarea.style.left = "-9999px";
  document.body.appendChild(textarea);
  textarea.select();
  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } catch (err) {
    console.error("Fallback copy failed:", err);
    alert("Could not copy to clipboard. Please copy manually.");
  } finally {
    document.body.removeChild(textarea);
  }
}
</script>
</details>


You should see one row per month, sorted oldest to newest. Confirm the columns are `Month`, `Monthly_Revenue`, and `Order_Count`.

Now wire up the visualization:

1. Back on the canvas, **Add a visualization** and drop the widget.
2. In the **Configuration Panel**, set:
   - **Title:** *checked*. Rename it to **Revenue and Orders by Month**.
   - **Visualization:** *Combo*
   - **X axis:** *Monthly Trend* > *Month*
   - **Y axis Bar:** *Monthly Trend* > *Monthly_Revenue*, Display Name *Revenue*
   - **Y axis Line:** *Monthly Trend* > *Order_Count*, Display Name *Orders*
3. Open the kebab `(⋮)` next to **Y axis** and toggle **Enable dual axis** so each series uses its own scale.
4. Open the kebab next to **Left Y axis (Bar)** → **Format → Custom → Type Currency ($)**.

### Task 4: Filter: slice the dashboard by `state`

1. In the bottom toolbar, click the **Filter** icon and drop the widget at the top of the canvas (above your two visualizations).
2. With the filter selected, in the configuration panel set:
   - **Filter type:** *Single value*
   - **Fields:** add **customers** > **state**.
3. Toggle **Title** on, then double-click the widget title and rename it to **State**.
4. Use the dropdown in the filter to pick a state, the **Customer Growth** counter should react. (The Combo Chart is built on `sales`, which doesn't include `state`, so it won't filter; that's expected. The marketing lead asked for a state filter on customer counts, which is what you've delivered.)

<details>
<summary>Hint, making the filter affect the chart too</summary>

If you want the Combo Chart to react to the state filter, you'd need to join <code>sales</code> to <code>customers</code> in the *Monthly Trend* dataset and add <code>customers.state</code> to the filter's <em>Fields</em>. That's a useful extension but not required by the spec, the marketing lead asked for state breakouts on customer counts, which the current setup delivers.

</details>

### Task 5: Publish the dashboard

Final UI steps:

1. Click **Publish** in the top-right of the dashboard.
2. Leave **Share data permission** selected (default).
3. Click **Publish** in the dialog.
4. In the **Sharing** dialog that opens, set the bottom dropdown to **Anyone in my account can view**, then close the dialog.
5. Use the context icon next to the dashboard name to flip to **View Published**. Confirm both visualizations render and the filter works.


<div style="max-width: 900px; margin: 0 auto; font-family: sans-serif;">
<div style="margin-top: 10px; padding: 18px 24px; background: #FFF6F4; border: 3px solid #FF5F46; border-radius: 10px;">
  <div style="font-size: 14pt; color: #0b2026; line-height: 1.6;">
    <div style="font-weight: 700; margin-bottom: 8px;">Nice work! You just:</div>
    <ul style="padding-left: 20px; margin: 0;">
      <li>Built a focused dashboard from a constrained business spec, not a feature dump, just what marketing asked for.</li>
      <li>Wrote a single-row SQL dataset for a Counter KPI with conditional formatting against a 1,000-customer goal.</li>
      <li>Wrote a monthly aggregation that drives a dual-axis Combo Chart on revenue and order count.</li>
      <li>Added a state Filter that lets stakeholders explore the customer count regionally, without touching the SQL.</li>
      <li>Published the dashboard with account-wide view access.</li>
    </ul>
    
  </div>
</div>
</div>


&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved.
Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
<script>
  document.getElementById("dbx-year").textContent = new Date().getFullYear();
</script>